# Homework

We study the family of Lecture 4,
$$
T_{\alpha,\beta}(x) = \beta - (1+\beta)|x|^{\alpha}, \qquad x \in [-1,1],
$$
with additive Gaussian noise of size $\sigma$ and periodic boundary conditions, and we fix $\beta = 1$ and
$$
\alpha = 3 + \frac{850}{1024} = 3.830078125 .
$$

The task is to prove that the Lyapunov exponent of this random system changes sign as the noise grows, by enclosing it at the two noise sizes
$$
\sigma_1 = \frac{1}{16} + \frac{15}{16}\cdot\frac{68}{1024}, \qquad
\sigma_2 = \frac{1}{16} + \frac{15}{16}\cdot\frac{70}{1024},
$$
and finding the first enclosure entirely positive and the second entirely negative. These two values are adjacent points of the grid used in Table 3 of [Galatolo, Lopez Vereau, Marangio, Nisoli](https://doi.org/10.1137/25M1786441), where the crossing between them is reported, so you can check your answer against theirs at the end.

Every step below is one of the steps of Lecture 4, in the same order and with the same functions; fill in the cells that end with `=`.

## Setting up

In [ ]:
import Pkg
Pkg.activate("./")
Pkg.add(["RigorousInvariantMeasures", "BallArithmetic", "IntervalArithmetic",
         "FFTW", "TaylorModels", "Plots", "RecipesBase", "LaTeXStrings"])

In [ ]:
using RigorousInvariantMeasures, IntervalArithmetic, BallArithmetic
using LinearAlgebra, FFTW, TaylorModels
using Plots, RecipesBase, LaTeXStrings

import IntervalArithmetic: inf, sup, mid, radius, diam, interval

setdisplay(:infsup; decorations = false, ng_flag = false)

Fix the parameters. Write $\alpha$ and the two noise sizes as exact fractions of intervals, not as decimals: they are grid points, and rounding them moves the Lyapunov exponent in the sixth digit.

We take `FFTNx` larger than the lecture did. The quadrature of the assembler converges like `FFTNx`$^{-2}$, and at $2^{14}$ it sits well below the other terms of the estimate.

In [ ]:
α = interval(3) + interval(850)/interval(1024)
β = interval(1)
K = 128
FFTNx = 2^14

σ₁ = interval(1)/interval(16) + (interval(15)/interval(16))*(interval(68)/interval(1024))
σ₂ = interval(1)/interval(16) + (interval(15)/interval(16))*(interval(70)/interval(1024))

(α, σ₁, σ₂)

Build the Fourier basis with `FourierAdjoint`, truncated at `K` and sampled at `FFTNx` points.

In [ ]:
B = 

The map lives on $[-1,1]$ and the basis on $[0,1]$, so we conjugate with the two changes of variables. Write `T_pm` for $T_{\alpha,\beta}$ on $[-1,1]$ and `T` for its conjugate on $[0,1]$.

In [ ]:
τ₁(x) = (x+1)/2          # from [-1, 1] to [0, 1]
τ₂(x) = 2*x-1            # from [0, 1] to [-1, 1]

T_pm(x) = 
T(x) = 

In [ ]:
plot(x -> IntervalArithmetic.mid(T(x)), 0, 1; label = "T on [0,1]")

## The operator

Assemble the deterministic transfer operator on the basis with `assemble`, and convert the matrix of intervals to a `BallMatrix`.

In [ ]:
@time PK = 
bPK = 
maximum(bPK.r)

Convolution with the Gaussian is diagonal in this basis, with entries $e^{-\sigma^2\pi^2k^2/2}$. We write the diagonal as a function of $\sigma$, so that we can use it twice, and we start with $\sigma_1$.

In [ ]:
NoiseInterval(σ, K) = Diagonal([[exp((-σ^2*interval(π)^2*interval(k)^2)/2) for k in 0:K];
                                [exp((-σ^2*interval(π)^2*interval(k)^2)/2) for k in -K:-1]])

σ = σ₁
bD  = 
PσK = 

## The stationary density

The stationary density is the eigenvector of eigenvalue $1$. Compute the eigendecomposition of the centre matrix, order the eigenvalues by decreasing modulus, and check that the first is $1$.

In [ ]:
F = 
p = sortperm(abs.(F.values), rev = true)
F.values[p][1:5]

Take the leading eigenvector, normalise it so that its first entry is $1$, which is the normalisation $\int f = 1$, and symmetrise it so that the density it represents is real.

In [ ]:
function symmetrise(v)
    w = zeros(eltype(v), length(v))
    N = (length(v)-1) ÷ 2
    w[1:N+1] = v[1:N+1]
    w[end-N+1:end] = [x' for x in reverse(v[2:N+1])]
    return w
end

fσK = 
fσK /= fσK[1]
fσKs = symmetrise(fσK)
bf = BallVector(fσKs)

In [ ]:
plot(real.(ifft(BallArithmetic.mid(bf))); label = "stationary density")

The vector `bf` is only an approximate fixed point. Compute its residual and the number $\varepsilon$ that bounds it, as in the lecture.

In [ ]:
res = 
ε   = 

## The mixing rate

Restrict the operator to the space of vectors of average zero, which is the complement of the first coordinate, and bound the norms of its powers with certified singular values.

In [ ]:
A = 

function power_norms(A, N)
    norms = zeros(N)
    Ai = A
    for i in 1:N
        norms[i] = BallArithmetic.svd_bound_L2_opnorm(Ai)
        Ai = Ai*A
    end
    return norms
end

@time norms = 

In [ ]:
plot(1:length(norms), norms; yscale = :log10, marker = :circle,
     label = "certified bound for the norm of the n-th power")

Read off the first power whose norm is below one, and call that norm $\eta$.

In [ ]:
n₁ = 
η  = 
n₁, η

## From the mixing rate to the density

The three constants below bound the error made by truncating the Fourier expansion at $K$; they are the ones of Lecture 4 and you do not need to derive them.

In [ ]:
Iπ = interval(π)

Γ  = sqrt(coth(1/(2*σ^2))/(σ*sqrt(Iπ)))*exp((-σ^2*interval(K)^2*Iπ^2)/2)
Γ¹ = (2/(σ^2*Iπ^2*interval(K)))*exp((-σ^2*interval(K)^2*Iπ^2)/2)
ρ₂ = sqrt(1/(2*σ*sqrt(Iπ)))*sqrt(coth(1/(2*σ^2)))

δ = Γ*(1+Γ¹) + ρ₂*Γ¹

Collect the constants $C_i$ of the estimate, which are the earlier norms raised to one where they are smaller, and assemble the $L^2$ bound
$$
\|f_{\sigma}-f_{\sigma,K,s}\|_{L^{2}} \leq \frac{1}{1-\eta}\sum_{i=0}^{n-1}C_i\big(\delta + \epsilon\big).
$$

In [ ]:
C = 
R = 

## The Lyapunov exponent

The exponent is $\lambda = \int \log|T'|\,f\,dm$, so we need the Fourier coefficients of $\log|T'|$. On $[-1,1]$,
$$
\log|T'_{\alpha,\beta}(x)| = \log\big((1+\beta)\alpha\big) + (\alpha-1)\log|x|,
$$
so everything reduces to the coefficients of $\log|x|$, which are $-\frac{1}{k\pi}\int_0^{k\pi}\frac{\sin t}{t}\,dt$ up to sign. The cell below encloses those integrals with Taylor models in `BigFloat`, exactly as in the lecture; run it as it is.

In [ ]:
setprecision(256)
Pi = interval(BigFloat, π)

sinc_over(t) = sin(t)/t
tay(i, x) = x^(2i+1)/(interval(BigFloat, factorial(big(2i+1)))*interval(BigFloat, 2i+1))

Nser = 60
I₀ = sum([(-1)^i*tay(i, Pi) for i in 0:Nser]) + interval(BigFloat, -1, 1)*abs(tay(Nser+1, Pi))

function int_over_arch(f, i; degree = 40)
    I = interval(inf(interval(BigFloat, i)*Pi), sup(interval(BigFloat, i+1)*Pi))
    m = interval(BigFloat, mid(I))
    tm = TaylorModel1(degree, m, I)
    return integrate(f(tm), 0)(interval(BigFloat, sup(I)) - m) -
           integrate(f(tm), 0)(interval(BigFloat, inf(I)) - m)
end

@time arcs = [I₀; [int_over_arch(sinc_over, i) for i in 1:K-1]]
maximum(diam.(arcs))

The cumulative sums give the coefficients. Note that `αb` and `βb` below are the same $\alpha$ and $\beta$ fixed at the top, carried into `BigFloat`; if you change the parameters, change them here too.

In [ ]:
coeff = cumsum(arcs) ./ [interval(BigFloat, i)*Pi for i in 1:K]
coeff01 = [(-1)^i for i in 1:K] .* coeff

αb = interval(BigFloat, 3) + interval(BigFloat, 850)/interval(BigFloat, 1024)
βb = interval(BigFloat, 1)

lnn = [log((1+βb)*αb) - (αb-1); -(αb-1)*[coeff01; reverse(coeff01)]]
length(lnn), lnn[1], lnn[2]

Pair the coefficients of the observable with those of the density; the real part is the value of the integral for the computed density.

In [ ]:
λK = 
real(λK)

The last ingredient turns the $L^2$ error on the density into an error on the integral, by Cauchy-Schwarz, with $\Upsilon$ the $L^2$ norm of the observable; it is given below. Assemble the enclosure of the Lyapunov exponent.

In [ ]:
Υ = sqrt(interval(BigFloat, 2))*((log((βb+1)*αb)-(αb-1))^2 + (αb-1)^2)^interval(BigFloat, 0.5)
Rb = interval(BigFloat, inf(R), sup(R))

λ₁ = 
(mid(λ₁), diam(λ₁))

Is the enclosure entirely on one side of zero? Which side?

In [ ]:
sup(λ₁) < 0, inf(λ₁) > 0

## The second noise size

Now repeat everything from the noise diagonal onwards with $\sigma = \sigma_2$. The basis, the map, the assembled operator `bPK` and the coefficients `lnn` of the observable do not depend on the noise, so they are not recomputed; the density, the mixing rate, the truncation constants and the error do.

The cleanest way is to wrap the steps you have just written into a function of $\sigma$ and call it twice; do that, and use it to produce the second enclosure.

In [ ]:
function lyapunov(σ)
    
end

λ₂ = lyapunov(σ₂)
(mid(λ₂), diam(λ₂))

In [ ]:
sup(λ₂) < 0, inf(λ₂) > 0

## What you have proved

Write here, in one or two sentences, what the two enclosures together say about the family at $\alpha = 3.830078125$ and $\beta = 1$, and at which of the two noise sizes the Lyapunov exponent is positive.

Then compare the two numbers with row 18 of Table 3 of [Galatolo, Lopez Vereau, Marangio, Nisoli](https://doi.org/10.1137/25M1786441), which reports the same two grid points. The signs agree and the values agree to five significant digits.

## If you have time

Two things worth trying, neither of which is needed for the proof above.

Take the two noise sizes closer together, at $j = 69$, and see whether the enclosure at that point still has a sign. The crossing lies between $j = 68$ and $j = 70$, and the enclosures are about $10^{-10}$ wide, so there is room to bisect a long way before the method runs out.

Repeat the whole computation at a different $\alpha$ from Table 3, for instance $\alpha = 3 + 512/1024$, whose crossing the paper reports near $\sigma = 0.297$.